In [ ]:
from google.colab import drive
import os
import shutil
import sys
from pathlib import Path

# ── Step 1: Mount Google Drive ────────────────────────────────────────────────
print("Mounting Google Drive...")
drive.mount("/content/drive", force_remount=True)

# ── Step 2: Clone code repo from GitHub ──────────────────────────────────────
# Check that src/ exists — a bare data/ folder from a previous session is NOT enough.
repo_dir = Path("/content/heritagelens")
if not (repo_dir / "src").exists():
    if repo_dir.exists():
        print("Repo dir exists but src/ is missing — re-cloning ...")
        shutil.rmtree(str(repo_dir))
    else:
        print("Cloning HeritageLens repo from GitHub...")
    os.system(
        "git clone -b Jaljala "
        "https://github.com/dhunganab2/heritagelens.git "
        "/content/heritagelens"
    )
    print("Repo cloned.")
else:
    print("Repo already present at /content/heritagelens (src/ found)")

# ── Step 3: Find and unzip dataset (heritagelens-data.zip or heritagelens.zip) ─
primary_zip  = "heritagelens-data.zip"   # RECOMMENDED: upload this to MyDrive
fallback_zip = "heritagelens.zip"        # Old full-project zip — also accepted

candidates = [
    Path("/content/drive/MyDrive") / primary_zip,
    Path("/content") / primary_zip,
    Path("/content/drive/MyDrive") / fallback_zip,
    Path("/content") / fallback_zip,
]

target_zip = None
for p in candidates:
    if p.exists():
        target_zip = p
        break

if target_zip:
    print(f"Found dataset zip: {target_zip}")
    print("Unzipping into /content/ ...")
    try:
        shutil.unpack_archive(str(target_zip), "/content/", "zip")
        print("Unzip complete.")
    except OSError as e:
        if getattr(e, "errno", None) == 107:
            print(
                "Transport endpoint error. "
                "Restart the runtime (Runtime > Restart session) and run this cell again."
            )
        raise
else:
    print("WARNING: No dataset zip found. Checked:")
    for p in candidates:
        print("  -", p)
    print("Upload 'heritagelens-data.zip' to Google Drive > MyDrive and rerun this cell.")

# ── Step 4: Move data/ into the repo if it landed outside ────────────────────
content_data = Path("/content/data")
repo_data    = repo_dir / "data"
if content_data.exists() and not repo_data.exists():
    print("Moving /content/data → /content/heritagelens/data ...")
    shutil.move(str(content_data), str(repo_data))
    print("Moved.")

# ── Step 5: Set ROOT, sys.path, and cwd ──────────────────────────────────────
ROOT = repo_dir if repo_dir.exists() else Path("/content")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

print(f"\nROOT = {ROOT}")
print("cwd  =", Path.cwd())
print("Contents:", sorted(x.name for x in ROOT.iterdir()))
print("\n✓ Ready — run the next cells in order.")

In [ ]:
# ── Cell 1: Install libraries & global setup ─────────────────────────────────
# ROOT and sys.path are set in Cell 0 — do NOT recompute them here.
!pip install -q transformers datasets torch torchvision tqdm nltk

import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from tqdm import tqdm

# Optional: Hugging Face login (needed only for gated models; gpt2 is public)
try:
    from google.colab import userdata
    from huggingface_hub import login
    try:
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            login(hf_token)
            print("Logged in to Hugging Face Hub.")
        else:
            print("No HF_TOKEN — using unauthenticated access (fine for gpt2).")
    except Exception:
        print("Using unauthenticated access to Hugging Face Hub (fine for gpt2).")
except Exception:
    pass

# ROOT was set by Cell 0; if this cell is run standalone, fall back gracefully.
if "ROOT" not in dir():
    ROOT = Path("/content/heritagelens")
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    import os; os.chdir(ROOT)

print(f"ROOT = {ROOT}")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Image transform (used in both training and inference)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# ── Cell 2: Dataset, DataLoader, Model, Optimizer ────────────────────────────
from torch.utils.data import random_split, DataLoader
from src.data.heritage_dataset import HeritageDataset

# ── Hyperparameters & paths ───────────────────────────────────────────────────
BATCH_SIZE   = 8      # Safe for T4 (16 GB VRAM); increase to 12 if no OOM
MAX_CAP_LEN  = 50     # Truncate / pad every caption to this many tokens
VAL_FRACTION = 0.10   # 10 % held out for validation
LR           = 3e-4
SEED         = 42

CKPT_DIR  = ROOT / "outputs" / "checkpoints"
FIG_DIR   = ROOT / "outputs" / "figures"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Load merged dataset (Wikimedia + DANAM) ───────────────────────────────────
full_dataset = HeritageDataset(
    json_path  = ROOT / "data/processed/metadata_merged.json",
    images_dir = [
        ROOT / "data/raw/wikimedia/images",
        ROOT / "data/raw/danam/images",
    ],
    transform=transform,   # defined in Cell 1
)
print(f"Total valid images: {len(full_dataset)}")

n_val   = int(len(full_dataset) * VAL_FRACTION)
n_train = len(full_dataset) - n_val
train_dataset, val_dataset = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
print(f"Split  →  train: {n_train}  |  val: {n_val}")

# ── Collate function ──────────────────────────────────────────────────────────
# HeritageDataset returns (image_tensor, caption_str).
# We tokenize here and build labels with -100 on padding tokens so GPT-2
# does not compute loss on padding positions.
# Batch shape: (images [B,3,224,224], input_ids [B,L], attention_mask [B,L], labels [B,L])

def collate_fn(samples):
    images   = torch.stack([s[0] for s in samples])
    captions = [s[1] for s in samples]

    encoded = tokenizer(
        captions,
        padding="max_length",
        truncation=True,
        max_length=MAX_CAP_LEN,
        return_tensors="pt",
    )
    input_ids      = encoded["input_ids"]       # [B, MAX_CAP_LEN]
    attention_mask = encoded["attention_mask"]  # [B, MAX_CAP_LEN]

    labels = input_ids.clone()
    labels[attention_mask == 0] = -100          # mask padding from loss

    return images, input_ids, attention_mask, labels

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)
print(f"DataLoaders  →  train batches: {len(train_loader)}  |  val batches: {len(val_loader)}")

# ── Model ─────────────────────────────────────────────────────────────────────
model = HeritageLens(vocab_size=tokenizer.vocab_size).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}  (encoder frozen)")

# Optimise only the trainable (non-frozen) parameters
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-4,
)

# Mixed-precision scaler for T4 FP16 training
scaler = torch.amp.GradScaler("cuda")

print("\nReady to train.")

In [ ]:
# --- 1. ENCODER (ResNet50) ---
class HeritageEncoder(nn.Module):
    def __init__(self):
        super(HeritageEncoder, self).__init__()
        resnet = models.resnet50(weights='ResNet50_Weights.DEFAULT')
        # Remove the last two layers to get spatial feature maps (7x7x2048)
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])

        # FREEZE ENCODER: As per project plan, we only train the decoder/attention
        for param in self.resnet.parameters():
            param.requires_grad = False

    def forward(self, images):
        features = self.resnet(images)  # [batch, 2048, 7, 7]
        features = features.permute(0, 2, 3, 1)  # [batch, 7, 7, 2048]
        features = features.view(features.size(0), -1, features.size(-1)) # [batch, 49, 2048]
        return features

# --- 2. ATTENTION (Bahdanau) ---
class HeritageAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super(HeritageAttention, self).__init__()
        self.W = nn.Linear(decoder_dim, attention_dim)
        self.U = nn.Linear(encoder_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1)

    def forward(self, features, hidden):
        # features: [batch, 49, 2048], hidden: [batch, 768]
        hidden_with_time = hidden.unsqueeze(1)
        score = torch.tanh(self.W(hidden_with_time) + self.U(features))
        weights = torch.softmax(self.v(score), dim=1)
        context = torch.sum(weights * features, dim=1)
        return context, weights

# --- 3. THE COMPLETE SYSTEM ---
class HeritageLens(nn.Module):
    def __init__(self, vocab_size):
        super(HeritageLens, self).__init__()
        self.encoder = HeritageEncoder()
        self.attention = HeritageAttention(encoder_dim=2048, decoder_dim=768, attention_dim=512)
        self.gpt2 = GPT2LMHeadModel.from_pretrained('gpt2')

        # Bridge to map visual context to GPT2 hidden size
        self.visual_projection = nn.Linear(2048, 768)

    def forward(self, images, captions, labels=None):
        # captions: input_ids [batch, seq_len]. labels: same shape, -100 for padding (optional)
        # Get visual features: [batch, 49, 2048]
        features = self.encoder(images)

        # For training, we use global average pooling for the initial visual context
        visual_context = self.visual_projection(features.mean(dim=1))

        # Get GPT2 word embeddings
        inputs_embeds = self.gpt2.transformer.wte(captions)

        # Inject visual context into the word embeddings
        combined_embeds = inputs_embeds + visual_context.unsqueeze(1)

        # Pass through GPT2 (use labels for loss so padding can be masked with -100)
        loss_labels = labels if labels is not None else captions
        outputs = self.gpt2(inputs_embeds=combined_embeds, labels=loss_labels)
        return outputs.loss, outputs.logits

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ── Configuration ─────────────────────────────────────────────────────────────
NUM_EPOCHS    = 20
history       = {"train_loss": [], "val_loss": [], "val_bleu": []}
best_val_loss = float("inf")
smoother      = SmoothingFunction().method1   # avoids zero BLEU on short captions

print(f"Starting training on T4 GPU  |  {NUM_EPOCHS} epochs  |  batch_size={BATCH_SIZE}")

for epoch in range(NUM_EPOCHS):

    # ── 1. TRAINING ───────────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")

    for batch in pbar:
        if batch is None:
            continue
        images, input_ids, _, labels = [x.to(device) for x in batch]

        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss, logits = model(images, input_ids, labels=labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_train_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    history["train_loss"].append(avg_train_loss)

    # ── 2. VALIDATION ─────────────────────────────────────────────────────────
    model.eval()
    total_val_loss = 0
    bleu_scores    = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            if batch is None:
                continue
            images, input_ids, _, labels = [x.to(device) for x in batch]

            with torch.amp.autocast("cuda"):
                loss, logits = model(images, input_ids, labels=labels)

            total_val_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)
            for i in range(len(preds)):
                clean_label_ids = labels[i][labels[i] != -100].tolist()
                reference = [tokenizer.decode(clean_label_ids, skip_special_tokens=True).split()]
                candidate = tokenizer.decode(preds[i], skip_special_tokens=True).split()
                if candidate:
                    score = sentence_bleu(
                        reference, candidate,
                        weights=(0.25, 0.25, 0.25, 0.25),
                        smoothing_function=smoother,
                    )
                    bleu_scores.append(score)

    avg_val_loss = total_val_loss / len(val_loader)
    avg_bleu     = float(np.mean(bleu_scores)) if bleu_scores else 0.0
    history["val_loss"].append(avg_val_loss)
    history["val_bleu"].append(avg_bleu)

    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS}  "
          f"Train Loss: {avg_train_loss:.4f}  "
          f"Val Loss: {avg_val_loss:.4f}  "
          f"BLEU-4: {avg_bleu:.4f}")

    # ── 3. CHECKPOINT ─────────────────────────────────────────────────────────
    ckpt_path = CKPT_DIR / f"epoch_{epoch+1:02d}_valloss_{avg_val_loss:.4f}.pt"
    torch.save({
        "epoch":           epoch + 1,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "val_loss":        avg_val_loss,
        "val_bleu":        avg_bleu,
        "train_loss":      avg_train_loss,
    }, ckpt_path)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), CKPT_DIR / "best_model.pt")
        print(f"  → Best model updated  (val_loss={best_val_loss:.4f})")

# ── 4. PLOT TRAINING CURVES ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(history["train_loss"], label="Train Loss")
ax1.plot(history["val_loss"],   label="Val Loss")
ax1.set_title("Model Loss (T4 Training)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(history["val_bleu"], color="green", label="BLEU-4")
ax2.set_title("Caption Quality (BLEU-4)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "training_curves.png", dpi=150)
plt.show()

print(f"\nTraining complete.")
print(f"Best val loss : {best_val_loss:.4f}")
print(f"Best model    : {CKPT_DIR / 'best_model.pt'}")

In [ ]:
def generate_caption(image_path, model, tokenizer, max_length=50):
    """Generate a caption for a heritage image using greedy decoding."""
    from PIL import Image
    model.eval()

    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return f"Error opening image: {e}"

    image_tensor = transform(image).unsqueeze(0).to(device)
    input_ids    = torch.tensor([[tokenizer.bos_token_id]]).to(device)
    generated    = []

    with torch.no_grad():
        for _ in range(max_length):
            _, logits = model(image_tensor, input_ids)
            next_id   = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(0)
            if next_id.item() == tokenizer.eos_token_id:
                break
            generated.append(next_id.item())
            input_ids = torch.cat([input_ids, next_id], dim=-1)

    return tokenizer.decode(generated, skip_special_tokens=True)

# ── Load the best checkpoint ──────────────────────────────────────────────────
best_ckpt = CKPT_DIR / "best_model.pt"
if best_ckpt.exists():
    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    print(f"Loaded best model from: {best_ckpt}")
else:
    print("No checkpoint found — using current model weights.")

# ── Pick a sample image from the dataset and caption it ──────────────────────
import glob, random

test_image = None
for pattern in [
    str(ROOT / "data/raw/wikimedia/images/**/*.jpg"),
    str(ROOT / "data/raw/danam/images/**/*.jpg"),
    str(ROOT / "data/raw/wikimedia/images/**/*.png"),
]:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        random.seed(42)
        test_image = random.choice(matches[:50])
        break

if test_image:
    print(f"\nTest image : {test_image}")
    caption = generate_caption(test_image, model, tokenizer)
    print(f"Caption    : {caption}")

    from PIL import Image
    import matplotlib.pyplot as plt
    img = Image.open(test_image)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Generated: {caption}", wrap=True, fontsize=10)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "sample_caption.png", dpi=120)
    plt.show()
else:
    print("No images found. Call: generate_caption('/path/to/image.jpg', model, tokenizer)")
